# Sketchy Parking Lot EDA

In [1]:
# Import and setup
import json
import os
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Domain parsing
import tldextract

# Text analysis
import re
from collections import Counter
from math import log2

## Import Data and Preparation

In [2]:
'''
recursively load data
'''

data_location = '../data/parked_suspicious/'

# Load summary
with open(f"{data_location}_summary.json") as f:
    summary = json.load(f)

dfs = []

# Iterate over summary labels to combine all JSONL files
for label in summary.keys():
    clean_label = label.replace(':', '-')

    filename = f"{data_location + clean_label}.jsonl"
    
    if not os.path.exists(filename):
        print(f"Missing: {filename}")
        continue
    
    try:
        df = pd.read_json(filename, lines=True)
        dfs.append(df)
        print(f"Loaded {filename} ({len(df)} rows)")
        
    except ValueError as e:
        print(f"Error reading {filename}: {e}")

# Combine all JSONL files into single DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

print("Total rows:", len(combined_df))
print("Services:", combined_df["service"].value_counts())

Loaded ../data/parked_suspicious/other-unclassified.jsonl (1000 rows)
Loaded ../data/parked_suspicious/other-empty-title.jsonl (1000 rows)
Loaded ../data/parked_suspicious/default-server.jsonl (1000 rows)
Loaded ../data/parked_suspicious/sedo.jsonl (1000 rows)
Loaded ../data/parked_suspicious/searchhounds.jsonl (1000 rows)
Loaded ../data/parked_suspicious/wix-unconnected.jsonl (1000 rows)
Loaded ../data/parked_suspicious/hugedomains.jsonl (1000 rows)
Loaded ../data/parked_suspicious/godaddy.jsonl (1000 rows)
Loaded ../data/parked_suspicious/afternic.jsonl (1000 rows)
Loaded ../data/parked_suspicious/expired-domain.jsonl (1000 rows)
Loaded ../data/parked_suspicious/atom.jsonl (1000 rows)
Loaded ../data/parked_suspicious/spaceship.jsonl (1000 rows)
Loaded ../data/parked_suspicious/other-domain-as-title.jsonl (642 rows)
Loaded ../data/parked_suspicious/dynadot.jsonl (619 rows)
Loaded ../data/parked_suspicious/brandbucket.jsonl (440 rows)
Loaded ../data/parked_suspicious/default-nginx.json

In [3]:
combined_df.head()

,url,domain,service,hops,redirectChain,finalUrl,title,textSnippet
0,http://aaa-fuerteventura-appartments.com,aaa-fuerteventura-appartments.com,other:unclassified,5,[{'from': 'https://aaa-fuerteventura-appartmen...,https://welcome.mywebsite-editor.com/render/ww...,Bald verfügbar,Bald verfügbar
1,http://a-afolienfahrzeuge.at,a-afolienfahrzeuge.at,other:unclassified,3,"[{'from': 'https://a-afolienfahrzeuge.at', 'to...",https://www.a-afolienfahrzeuge.at/,Connect Your Domain,Connect Your Domain You're almost done! Your c...
2,http://a-lola-estilo.com,a-lola-estilo.com,other:unclassified,3,"[{'from': 'https://a-lola-estilo.com', 'to': '...",https://www.a-lola-estilo.com/password,A Lola Estilo – Abertura em breve,A Lola Estilo – Abertura em breve A Lola Estil...
3,http://a-petrenko.com,a-petrenko.com,other:unclassified,3,"[{'from': 'https://a-petrenko.com', 'to': 'htt...",https://a-petrenko.com/info/,a-petrenko,a-petrenko
4,http://a-point-of-view.de,a-point-of-view.de,other:unclassified,3,"[{'from': 'https://a-point-of-view.de', 'to': ...",https://p2m-1.forsaledomain.net/true2.php?id=a...,Just a moment...,Just a moment... p2m-1.forsaledomain.net Verif...


In [4]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15673 entries, 0 to 15672
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   url            15673 non-null  object
 1   domain         15673 non-null  object
 2   service        15673 non-null  object
 3   hops           15673 non-null  int64 
 4   redirectChain  15673 non-null  object
 5   finalUrl       15673 non-null  object
 6   title          15673 non-null  object
 7   textSnippet    15673 non-null  object
dtypes: int64(1), object(7)
memory usage: 979.7+ KB


In [5]:
combined_df.describe()

,hops
count,15673.000000
mean,0.710202
std,0.998587
min,0.000000
25%,0.000000
50%,0.000000
75%,2.000000
max,5.000000


## Text Analysis for Identification of Parked or Ad Domains 

In [6]:
'''
combine, clean, and tokenize
'''
text_cols = ["title", "textSnippet"]

def preprocess_text(row, text_cols):
    if not text_cols:
        return "", []
    
    combined = " ".join([str(row[col]) for col in text_cols if pd.notna(row[col])])
    # will explore other tokenization methods 
    combined = combined.strip().lower()
    combined = re.sub(r"http\S+|www\.\S+", " ", combined)
    combined = re.sub(r"[^a-z0-9\s]", " ", combined)
    combined = re.sub(r"\s+", " ", combined).strip()
    
    tokens = combined.split()
    
    return combined, tokens

combined_df[["clean_text", "tokens"]] = combined_df.apply(
    lambda row: pd.Series(preprocess_text(row, text_cols)),
    axis=1
)

combined_df[["clean_text", "tokens"]].head()

,clean_text,tokens
0,bald verf gbar bald verf gbar,"[bald, verf, gbar, bald, verf, gbar]"
1,connect your domain connect your domain you re...,"[connect, your, domain, connect, your, domain,..."
2,a lola estilo abertura em breve a lola estilo ...,"[a, lola, estilo, abertura, em, breve, a, lola..."
3,a petrenko a petrenko,"[a, petrenko, a, petrenko]"
4,just a moment just a moment p2m 1 forsaledomai...,"[just, a, moment, just, a, moment, p2m, 1, for..."


In [ ]:
'''
testing these keywords for now
'''
ad_keywords = {
    "ad", "ads", "advertisement", "sponsored", "promotion",
    "click", "offer", "sale", "buy", "discount"
}

parked_keywords = {
    "domain for sale", "buy this domain", "this domain is for sale",
    "parked", "coming soon", "under construction", "related links", "related searches", 
    "search results", "powered by", "ads by", "privacy policy", "this webpage was generated",
    "domain parked free", "listing expired"
}

suspicious_keywords = {
    "virus", "malware", "warning", "security alert", "download now", "install", 
    "verify", "urgent", "your computer", "system alert", "scan now", "risk detected",
    "click here to continue", "enable notifications"
}

In [8]:
'''
count keyword matches
'''
def keyword_features(text, tokens):
    token_set = set(tokens)
    
    def count_hits(keyword_set):
        hits = 0
        for kw in keyword_set:
            if " " in kw:
                if kw in text:
                    hits += 1
            else:
                if kw in token_set:
                    hits += 1
        return hits
    
    return pd.Series({
        "ad_keyword_hits": count_hits(ad_keywords),
        "parked_keyword_hits": count_hits(parked_keywords),
        "suspicious_keyword_hits": count_hits(suspicious_keywords)
    })

combined_df[[
    "ad_keyword_hits",
    "parked_keyword_hits",
    "suspicious_keyword_hits"
]] = combined_df.apply(
    lambda row: keyword_features(row["clean_text"], row["tokens"]),
    axis=1
)

In [9]:
for col in ["ad", "parked", "suspicious"]:
    combined_df[f"contains_{col}_text"] = (combined_df[f"{col}_keyword_hits"] > 0).astype(int)

In [10]:
for col in ["ad_keyword_hits", "parked_keyword_hits", "suspicious_keyword_hits"]:
    print(f"\nTop rows for {col}")
    display(
        combined_df.sort_values(col, ascending=False)[
            ["clean_text", col]
        ].head(5)
    )


Top rows for ad_keyword_hits


,clean_text,ad_keyword_hits
11915,body click for sale spaceship com body click f...,4
7610,access discount access discount excellent 4 6 ...,4
3154,ad academy de is available for purchase sedo c...,4
3155,ad min de is available for purchase sedo com a...,4
3630,anie net is available for purchase sedo com an...,3



Top rows for parked_keyword_hits


,clean_text,parked_keyword_hits
15263,abrahamczik de this website is for sale abraha...,3
12307,aajagency com aajagency com buy this domain 20...,2
1383,aall electric com is parked free courtesy of g...,2
1382,aalmubasher org is parked free courtesy of god...,2
1376,aaledaart com is parked free courtesy of godad...,2



Top rows for suspicious_keyword_hits


,clean_text,suspicious_keyword_hits
384,default page default page you are all set to g...,1
12147,about fr about fr about fr click here to continue,1
639,default page default page you are all set to g...,1
1377,warning undefined array key auto css in custom...,1
132,a j wright fibre install professionals a j wri...,1


In [11]:
combined_df[
    [
        "contains_ad_text",
        "contains_parked_text",
        "contains_suspicious_text"
    ]
].mean()

contains_ad_text            0.440630
contains_parked_text        0.218210
contains_suspicious_text    0.003254
dtype: float64

## Data Wrangling

Feature engineer

In [6]:
'''
extract TLD, domain and subdomain using tldextract

Parse the full domain for analysis.
'''
# change domain to domain_full to preserve original domain for tldextract
combined_df["domain_full"] = combined_df["domain"]
combined_df.drop("domain", axis=1, inplace=True)

def extract_domain_parts(domain):
    ext = tldextract.extract(domain)
    return pd.Series({
        "subdomain": ext.subdomain,
        "domain_name": ext.domain,
        "tld": ext.suffix
    })

combined_df[["subdomain", "domain_name", "tld"]] = combined_df["domain_full"].apply(extract_domain_parts)

In [7]:
'''
domain length feature

Identify how many characters the domain has.
'''
combined_df["len_domain_full"] = combined_df["domain_full"].apply(len)

In [8]:
'''
is_suspicious feature

This feature is derived from the amount of hops (redirects) domain has.
If a domain has more than 2 hops, we can consider it suspicious.
Subject to change based on analysis of hops distribution.
'''
threshold_hops = 2
combined_df["is_suspicious"] = combined_df["hops"].apply(lambda x: 1 if x > threshold_hops else 0)

In [9]:
'''
Shannons entropy of the domain feature

Calculate the Shannon entropy of the domain name to measure its randomness.
Higher entropy may indicate a more complex and potentially suspicious domain.
'''


'\nShannons entropy of the domain feature\n\nCalculate the Shannon entropy of the domain name to measure its randomness.\nHigher entropy may indicate a more complex and potentially suspicious domain.\n'

In [10]:
combined_df.head()

,url,service,hops,redirectChain,finalUrl,title,textSnippet,domain_full,subdomain,domain_name,tld,len_domain_full,is_suspicious
0,http://aaa-fuerteventura-appartments.com,other:unclassified,5,[{'from': 'https://aaa-fuerteventura-appartmen...,https://welcome.mywebsite-editor.com/render/ww...,Bald verfügbar,Bald verfügbar,aaa-fuerteventura-appartments.com,,aaa-fuerteventura-appartments,com,33,1
1,http://a-afolienfahrzeuge.at,other:unclassified,3,"[{'from': 'https://a-afolienfahrzeuge.at', 'to...",https://www.a-afolienfahrzeuge.at/,Connect Your Domain,Connect Your Domain You're almost done! Your c...,a-afolienfahrzeuge.at,,a-afolienfahrzeuge,at,21,1
2,http://a-lola-estilo.com,other:unclassified,3,"[{'from': 'https://a-lola-estilo.com', 'to': '...",https://www.a-lola-estilo.com/password,A Lola Estilo – Abertura em breve,A Lola Estilo – Abertura em breve A Lola Estil...,a-lola-estilo.com,,a-lola-estilo,com,17,1
3,http://a-petrenko.com,other:unclassified,3,"[{'from': 'https://a-petrenko.com', 'to': 'htt...",https://a-petrenko.com/info/,a-petrenko,a-petrenko,a-petrenko.com,,a-petrenko,com,14,1
4,http://a-point-of-view.de,other:unclassified,3,"[{'from': 'https://a-point-of-view.de', 'to': ...",https://p2m-1.forsaledomain.net/true2.php?id=a...,Just a moment...,Just a moment... p2m-1.forsaledomain.net Verif...,a-point-of-view.de,,a-point-of-view,de,18,1


## Classify Based on Engineered Features

Classify as parked, expired, default, or active domain.

## Data Distributions and Visualizations

In [11]:
combined_df.columns

Index(['url', 'service', 'hops', 'redirectChain', 'finalUrl', 'title',
       'textSnippet', 'domain_full', 'subdomain', 'domain_name', 'tld',
       'len_domain_full', 'is_suspicious'],
      dtype='object')

## Save JSONL

In [12]:
# save as JSONL for future use

# combined_df.to_json('data/parked_suspicious/_combined_data.jsonl', orient='records', lines=True)